# Project Lab 2: Quality, Streaming, Star Schemas and Storage

Notebook for **Chapter 16** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 Project: Data-Quality Framework with Alerts

In [ ]:
import sqlite3, json
from dataclasses import dataclass

@dataclass
class Check:
    name: str; sql: str; severity: str
    threshold: float = 0

def run_checks(db, checks, alert):
    results = []
    for c in checks:
        bad = db.execute(c.sql).fetchone()[0]
        ok = bad <= c.threshold
        results.append({"check": c.name, "bad": bad, "ok": ok, "severity": c.severity})
        if not ok: alert(c, bad)
    return results

alerts = []
def alert(c, bad): alerts.append(f"[{c.severity.upper()}] {c.name}: {bad} bad (allowed {c.threshold})")

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE appointments(appt_id TEXT, patient_id TEXT, wait_min REAL, dept TEXT, booked_at TEXT);
INSERT INTO appointments VALUES
('A1','P1',15,'ENT','2024-06-01T09:00:00'),('A2','P2',40,'ENT','2024-06-01T09:30:00'),
('A3','P3',-3,'CARDIO','2024-06-01T10:00:00'),('A3','P3',25,'CARDIO','2024-06-01T10:00:00'),
('A4',NULL,500,'XRAY','2024-06-01T10:30:00'),('A5','P5',30,'ENT','2024-06-01T11:00:00');""")
checks = [
 Check("appt_id unique", "SELECT COUNT(*)-COUNT(DISTINCT appt_id) FROM appointments", "error"),
 Check("patient_id not null", "SELECT COUNT(*) FROM appointments WHERE patient_id IS NULL", "error"),
 Check("wait_min in 0..240", "SELECT COUNT(*) FROM appointments WHERE wait_min<0 OR wait_min>240", "warn"),
 Check("dept in known list", "SELECT COUNT(*) FROM appointments WHERE dept NOT IN ('ENT','CARDIO','XRAY')", "error"),
 Check("row count >= 5", "SELECT CASE WHEN COUNT(*)>=5 THEN 0 ELSE 1 END FROM appointments", "error"),
 Check("null rate patient_id <= 20%", "SELECT AVG(patient_id IS NULL) FROM appointments", "warn", threshold=0.20),
]
res = run_checks(db, checks, alert)
for r in res: print(("PASS" if r["ok"] else "FAIL"), r["check"], r["bad"])
print("\nALERTS:"); print("\n".join(alerts))
errors = [r for r in res if not r["ok"] and r["severity"] == "error"]
print("\npipeline decision:", "STOP (publish blocked)" if errors else "PUBLISH")
json.dump(res, open("dq_report.json", "w"), indent=1)

## 16.2 Project: Streaming Windowed Aggregation with Late Events

In [ ]:
from collections import defaultdict
WINDOW, LATENESS = 60, 30
events = [(5,100),(20,50),(65,200),(50,70),(70,30),(125,10),(62,40),(10,999),(130,60)]
windows = defaultdict(float); dropped = []; watermark = 0
for et, amt in events:
    watermark = max(watermark, et - LATENESS)
    w_start = et // WINDOW * WINDOW
    if w_start + WINDOW <= watermark:
        dropped.append((et, amt))
        print(f"event t={et:3d} amt={amt:3d}  watermark={watermark:3d}  DROPPED (window closed)")
        continue
    windows[w_start] += amt
    print(f"event t={et:3d} amt={amt:3d}  watermark={watermark:3d}  -> window [{w_start},{w_start+WINDOW})")
print("\nresult:", dict(sorted(windows.items())))
print("dropped:", dropped)

In [ ]:
from pyspark.sql import functions as F
agg = (events.withWatermark("event_time", "30 seconds")
       .groupBy(F.window("event_time", "1 minute")).agg(F.sum("amount")))

## 16.3 Project: Warehouse Star Schema with SCD2 for an Online Shop

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE dim_customer(cust_key INTEGER PRIMARY KEY AUTOINCREMENT, cust_id TEXT, city TEXT,
   valid_from TEXT, valid_to TEXT, is_current INT);
CREATE TABLE dim_product(prod_key INTEGER PRIMARY KEY, name TEXT, category TEXT);
CREATE TABLE fact_sales(order_id TEXT, cust_key INT, prod_key INT, order_date TEXT, qty INT, amount REAL);
INSERT INTO dim_product VALUES(1,'Kettle','Home'),(2,'Notebook','Stationery');""")

def scd2_upsert(cust_id, city, as_of):
    cur = db.execute("SELECT cust_key, city FROM dim_customer WHERE cust_id=? AND is_current=1", (cust_id,)).fetchone()
    if cur and cur[1] == city: return "unchanged"
    if cur: db.execute("UPDATE dim_customer SET valid_to=?, is_current=0 WHERE cust_key=?", (as_of, cur[0]))
    db.execute("INSERT INTO dim_customer(cust_id,city,valid_from,valid_to,is_current) VALUES(?,?,?,'9999-12-31',1)", (cust_id, city, as_of))
    return "changed" if cur else "new"

def key_for(cust_id, order_date):
    return db.execute("SELECT cust_key FROM dim_customer WHERE cust_id=? AND valid_from<=? AND ?<valid_to",
                      (cust_id, order_date, order_date)).fetchone()[0]

print(scd2_upsert("C1", "Pune", "2024-01-01"), scd2_upsert("C1", "Pune", "2024-02-01"), scd2_upsert("C1", "Delhi", "2024-03-01"))
for oid, d, p, q, a in [("O1","2024-02-10",1,1,800.0),("O2","2024-03-15",2,3,150.0),("O3","2024-03-20",1,2,1600.0)]:
    db.execute("INSERT INTO fact_sales VALUES(?,?,?,?,?,?)", (oid, key_for("C1", d), p, d, q, a))
print(db.execute("SELECT * FROM dim_customer").fetchall())
print(db.execute("""SELECT c.city, SUM(f.amount) FROM fact_sales f JOIN dim_customer c USING(cust_key)
                    GROUP BY c.city ORDER BY c.city""").fetchall())
print(db.execute("""SELECT p.category, c.city, SUM(f.amount), SUM(f.qty) FROM fact_sales f
   JOIN dim_customer c USING(cust_key) JOIN dim_product p USING(prod_key) GROUP BY 1,2 ORDER BY 1,2""").fetchall())
print("orphans:", db.execute("SELECT COUNT(*) FROM fact_sales f LEFT JOIN dim_customer c USING(cust_key) WHERE c.cust_key IS NULL").fetchone()[0])
print("current rows per customer:", db.execute("SELECT cust_id, SUM(is_current) FROM dim_customer GROUP BY cust_id").fetchall())

## 16.4 Project: Parquet versus CSV, Partitioning and Compression Benchmark

In [ ]:
import random, zlib, time, struct
N = 200_000
rnd = random.Random(3); cities = ["Pune","Delhi","Chennai","Mumbai","Kolkata"]
rows = [(i, rnd.choice(cities), rnd.randint(20,120), round(rnd.uniform(50,2000),2)) for i in range(N)]
csv_bytes = "\n".join(",".join(map(str, r)) for r in rows).encode()
cols = list(zip(*rows))
col_bytes = [struct.pack(f"{N}q", *cols[0]), "\n".join(cols[1]).encode(),
             struct.pack(f"{N}H", *cols[2]), struct.pack(f"{N}d", *cols[3])]
names = ["order_id","city","minutes","amount"]
z = lambda b: len(zlib.compress(b, 6))
print(f"{'layout':28s}{'raw KB':>10s}{'zlib KB':>10s}")
print(f"{'row-wise CSV':28s}{len(csv_bytes)/1024:10.0f}{z(csv_bytes)/1024:10.0f}")
print(f"{'column-wise (binary)':28s}{sum(map(len,col_bytes))/1024:10.0f}{sum(map(z,col_bytes))/1024:10.0f}")
for n, b in zip(names, col_bytes): print(f"   column {n:10s}{len(b)/1024:10.0f}{z(b)/1024:10.0f}")
t=time.perf_counter(); parts=[l.split(",")[2] for l in csv_bytes.decode().split("\n")]
avg_csv=sum(map(int,parts))/N; t_csv=time.perf_counter()-t
t=time.perf_counter(); arr=struct.unpack(f"{N}H", zlib.decompress(zlib.compress(col_bytes[2])))
avg_col=sum(arr)/N; t_col=time.perf_counter()-t
print(f"avg minutes csv={avg_csv:.3f} ({t_csv:.3f}s)  columnar={avg_col:.3f} ({t_col:.3f}s)")
by_city = {c: [r for r in rows if r[1]==c] for c in cities}
print("rows scanned for city=Pune: unpartitioned", N, "| partitioned", len(by_city["Pune"]))

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.csv as pc
t = pc.read_csv("orders.csv")
pq.write_table(t, "orders_snappy.parquet", compression="snappy")
pq.write_to_dataset(t, root_path="orders_by_city", partition_cols=["city"])
avg = pq.read_table("orders_snappy.parquet", columns=["minutes"])

## 16.5 Break It on Purpose: Three Drills and a Check Set

For each lab in this chapter, here is one way it fails quietly, the real output that shows it, and the check that would have caught it. Reproduce each drill in your own version of the lab, then keep the check.

In [ ]:
# Drill: SCD2 validity check. Good history has no gaps and no overlaps per customer.
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE dim(customer_id TEXT, city TEXT, valid_from TEXT, valid_to TEXT, is_current INTEGER);
INSERT INTO dim VALUES ('c1','Pune','2025-01-01','2025-03-01',0),
                       ('c1','Delhi','2025-03-01','9999-12-31',1),
                       ('c2','Pune','2025-01-01','2025-02-15',0),
                       ('c2','Goa','2025-02-10','9999-12-31',1);   -- bug: overlaps the previous row by 5 days
""")
q_overlap = """SELECT a.customer_id, a.valid_to, b.valid_from FROM dim a JOIN dim b
               ON a.customer_id=b.customer_id AND a.valid_from < b.valid_from AND a.valid_to > b.valid_from"""
q_gap = """SELECT a.customer_id, a.valid_to, b.valid_from FROM dim a JOIN dim b
           ON a.customer_id=b.customer_id AND a.valid_to < b.valid_from
           AND NOT EXISTS (SELECT 1 FROM dim c WHERE c.customer_id=a.customer_id AND c.valid_from=a.valid_to)"""
q_current = "SELECT customer_id, SUM(is_current) FROM dim GROUP BY customer_id HAVING SUM(is_current) <> 1"
print("overlaps:", db.execute(q_overlap).fetchall())
print("gaps:", db.execute(q_gap).fetchall())
print("customers without exactly one current row:", db.execute(q_current).fetchall())

overlaps: [('c2', '2025-02-15', '2025-02-10')]
gaps: []
customers without exactly one current row: []


In [ ]:
# Drill: a late event beyond the watermark. Count what the window result says versus the truth.
events = [(5, "a"), (20, "b"), (65, "c"), (70, "d"), (12, "late1"), (130, "e"), (40, "late2")]  # (event time s, id) in ARRIVAL order
WINDOW, LATENESS = 60, 30
wm = -1
counts, dropped = {}, []
for t, i in events:
    wm = max(wm, t - LATENESS)               # watermark = max event time seen minus allowed lateness
    w = t // WINDOW
    if (w + 1) * WINDOW <= wm:               # window already closed
        dropped.append(i); continue
    counts[w] = counts.get(w, 0) + 1
print("window counts:", counts)
print("dropped as too late:", dropped)
truth = {}
for t, i in events: truth[t // WINDOW] = truth.get(t // WINDOW, 0) + 1
print("truth counts :", truth)

window counts: {0: 3, 1: 2, 2: 1}
dropped as too late: ['late2']
truth counts : {0: 4, 1: 2, 2: 1}


In [ ]:
# Drill: a benchmark that lies. Run each measurement three times and report the median, and check rows match.
import csv, io, json, statistics, time, random
random.seed(2)
rows = [{"id": i, "city": random.choice(["Pune", "Delhi", "Goa"]), "amount": round(random.random() * 100, 2)} for i in range(50000)]
buf = io.StringIO(); w = csv.DictWriter(buf, fieldnames=["id", "city", "amount"]); w.writeheader(); w.writerows(rows)
text_csv = buf.getvalue(); text_json = "\n".join(json.dumps(r) for r in rows)
def total_csv(): return round(sum(float(r["amount"]) for r in csv.DictReader(io.StringIO(text_csv))), 2)
def total_json(): return round(sum(json.loads(l)["amount"] for l in text_json.split("\n")), 2)
def timed(f):
    ts = []
    for _ in range(3):
        t0 = time.perf_counter(); out = f(); ts.append(time.perf_counter() - t0)
    return out, statistics.median(ts)
a, ta = timed(total_csv); b, tb = timed(total_json)
print("same answer from both formats:", a == b, a)
print("bytes csv:", len(text_csv), " bytes jsonl:", len(text_json), " ratio:", round(len(text_json) / len(text_csv), 2))

same answer from both formats: True 2491504.65
bytes csv: 878748  bytes jsonl: 2328731  ratio: 2.65


## Chapter recap

## Practice set

**1.** (Easy) A check returns 3 bad rows, threshold 0, severity warn. Does the pipeline stop? **Answer:** No. A warn alerts but continues; only error severity stops it.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch16.html